In [1]:
import json
import pandas as pd
from utils import compare_preds

In [2]:
ENTITIES_TO_PREDICT = ["HouseNumber", "StreetName", "City", "District", "State", "Country"]

csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)

# Load cached predictions
with open("cached_preds_val.json", "r") as f:
    cached_preds = json.load(f)

In [3]:
# Focus on a single config instead of best-per-family
CONFIG_NAME = "Llama-3-8B-prompt2-similar3shot"

In [4]:
# Compute metrics for the selected config only
if CONFIG_NAME not in cached_preds:
    raise KeyError(f"Config '{CONFIG_NAME}' not found in cached_preds. Available keys: {list(cached_preds.keys())[:5]} ...")

preds = cached_preds[CONFIG_NAME]["preds"]
preds_df = pd.DataFrame(preds)
metrics = compare_preds(preds_df, bzkopen_val, target_columns=ENTITIES_TO_PREDICT)
print(f"Metrics for {CONFIG_NAME}:")
print(metrics)

Metrics for Llama-3-8B-prompt2-similar3shot:
OrderedDict({'accuracy': np.float64(0.36973684210526314), 'precision': np.float64(0.8413173652694611), 'recall': np.float64(0.36973684210526314), 'f1': np.float64(0.5137111517367459), 'accuracy_with_tol_1': np.float64(0.37236842105263157), 'accuracy_with_tol_2': np.float64(0.3828947368421053), 'accuracy_with_tol_3': np.float64(0.9460526315789474), 'accuracy_with_tol_4': np.float64(0.9552631578947368), 'average_levenshtein': np.float64(2.1092105263157896), 'average_similarity': np.float64(0.39228940779993826), 'average_levenshtein_match': np.float64(4.932307692307693), 'average_similarity_match': np.float64(0.9173536920860095), 'no_match_rate': np.float64(0.5723684210526316)})


In [5]:
# Helper to show all error cases for a specific config/entity

def show_errors_for_config_entity(config_name, entity):
    """Display prediction errors for a given config and entity."""
    if config_name not in cached_preds:
        print(f"Config '{config_name}' not found in cached_preds.")
        return

    preds = cached_preds[config_name]["preds"]
    preds_df = pd.DataFrame(preds)

    if entity not in preds_df.columns:
        print(f"Entity '{entity}' not found in predictions.")
        return

    mismatch_mask = preds_df[entity] != bzkopen_val[entity].values
    error_cases = pd.DataFrame({
        "Address": bzkopen_val.loc[mismatch_mask, "FullAddress"].values,
        "Predicted": preds_df.loc[mismatch_mask, entity].values,
        "Actual": bzkopen_val.loc[mismatch_mask, entity].values,
    })

    print(f"\nErrors for {config_name} / {entity}:")
    print(error_cases.to_string(index=False))
    print(f"\nTotal errors: {len(error_cases)}")
    return error_cases

In [6]:
# Wrapper: use CONFIG_NAME by default

def show_errors_for_selected_config(entity):
    return show_errors_for_config_entity(CONFIG_NAME, entity)

In [20]:
df = show_errors_for_selected_config("State")


Errors for Llama-3-8B-prompt2-similar3shot / State:
                                              Address     Predicted Actual
                            Regensburg, Königstr. 2/I           NaN    NaN
                                             Dortmund           NaN    NaN
                        Nürnberg, Nibelungenstrasse 8           NaN    NaN
                         Jöhlingen/Krs.Durlach/Baden.   Krs.Durlach    NaN
            8 Burlington Road, Manchester 20/England.           NaN    NaN
                                    Leer/Ostfriesland           NaN    NaN
                                             Hannover           NaN    NaN
                    New York, USA, 220 West 98th Str.           NaN    NaN
                                               Berlin           NaN    NaN
Berlin-Lichterfelde-Ost, Berlinerstr.48a (Altersheim)           NaN    NaN
                                              Krefeld           NaN    NaN
                            Nürschan, Bez.Mies/

In [21]:
df

,Address,Predicted,Actual
0,"Regensburg, Königstr. 2/I",NaN,NaN
1,Dortmund,NaN,NaN
2,"Nürnberg, Nibelungenstrasse 8",NaN,NaN
3,Jöhlingen/Krs.Durlach/Baden.,Krs.Durlach,NaN
4,"8 Burlington Road, Manchester 20/England.",NaN,NaN
...,...,...,...
140,Sosnowice/Polen,NaN,NaN
141,2114-79 St. Jackson Heights. N.Y. USA,N.Y.,NaN
142,Losone CSR,NaN,NaN
143,Rum.,NaN,NaN


In [4]:
# Single-address test with Llama-3-8B-prompt2-similar3shot

from mllms import SimilarExamples, JsonDictPromptTemplate
import pandas as pd
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#  Load training data used for SimilarExamples
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])
bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)

#  Prompt 2 definition
llama_prompt2 = JsonDictPromptTemplate(
    "You are a german archivist handling the digitalization of german documents from the compensation efforts that followed the second world war. "
    "Your current task consists of annotating addresses found in the archival documents, identifying the respective components of each address. "
    "Consider the component types: " + ", ".join(ENTITIES_TO_PREDICT + ["Other"]) + ". "
    "It is essential that you remain loyal to the original text and do not add any information not explictly mentioned in the address. "
    "Addresses will most times be written in german, meaning country and city names may be in german. The addresses may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\", \"avenue\" or its abbreviation \"str.\" and \"av.\" for street.\n"
    "These terms may occur as a suffix to another word.\n"
    "Format the output as a JSON object with the component types as keys.\n%(examples)s"
    "Now annotate the following address:\n%(address)s"
)

# 4) SimilarExamples config for "similar3shot" (num_examples=3)
similar3shot_cfg = {
    "factory": SimilarExamples,
    "factory_kargs": dict(
        example_addresses=bzkopen_train["FullAddress"],
        example_labels=bzkopen_train,
        num_examples=3,
        labels_to_include=ENTITIES_TO_PREDICT,
        device=device,
    ),
}


c:\Users\amg\bzk-address-parsing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

from augmentation import GeoNamesLookup, augment_dataset

GEONAMES_USER = "amelgd"  

geo = GeoNamesLookup(username=GEONAMES_USER)

# Country values in the dataframe are resolved automatically via GeoNames.
augmented = augment_dataset(
    df         = bzkopen_train,
    geo        = geo,
    n_augments = 3,
    seed       = 42,
)

has_label = bzkopen_train["City"].notna() | bzkopen_train["State"].notna()
print(f"Original rows with City/State labels : {has_label.sum()}")
print(f"New synthetic rows generated         : {len(augmented)}")

augmented[["FullAddress", "City", "State", "Country", "is_augmented", "source_address"]]


Fetching pools for 'USA' (US) …
  → 1000 cities, 60 states
Fetching pools for 'Polen' (PL) …
  → 1000 cities, 16 states
Fetching pools for 'Israel' (IL) …
  → 1000 cities, 9 states
Fetching pools for 'Griechenland' (GR) …
  → 1000 cities, 14 states
Fetching pools for 'Frankreich' (FR) …
  → 1000 cities, 17 states
Fetching pools for 'Schweiz' (CH) …
  → 1000 cities, 26 states
Fetching pools for 'Italien' (IT) …
  → 1000 cities, 20 states
Fetching pools for 'Rumänien' (RO) …
  → 1000 cities, 42 states
Fetching pools for 'Ungarn' (HU) …
  → 1000 cities, 20 states
Fetching pools for 'Russland' (RU) …
  → 1000 cities, 83 states
Fetching pools for 'CSR' (CZ) …
  → 1000 cities, 14 states
Fetching pools for 'Ukraine' (UA) …
  → 1000 cities, 27 states
Fetching pools for 'LATVIA' (LV) …
  → 1000 cities, 43 states
Fetching pools for 'Litauen' (LT) …
  → 1000 cities, 10 states
Original rows with City/State labels : 720
New synthetic rows generated         : 1257


,FullAddress,City,State,Country,is_augmented,source_address
0,Ruskowa,Ruskowa,NaN,NaN,False,NaN
1,"170 Clymer Street, Brooklyn, N.Y. USA",N.Y.,NaN,USA,False,NaN
2,Drohobycz,Drohobycz,NaN,NaN,False,NaN
3,"London N.W.3/England, 109 Green Hill",London,NaN,NaN,False,NaN
4,Dresden,Dresden,NaN,NaN,False,NaN
...,...,...,...,...,...,...
1252,Slušovice / CSR,Slušovice,NaN,CSR,True,Pilzen / CSR
1253,Miroslav / CSR,Miroslav,NaN,CSR,True,Pilzen / CSR
1254,"311 Lincoln Rd., Koreatown, Midway Islands/USA",Koreatown,Midway Islands,USA,True,"311 Lincoln Rd., Miami Beach, Florida/USA"
1255,"311 Lincoln Rd., Wilmington, Bunker's Shoal/USA",Wilmington,Bunker's Shoal,USA,True,"311 Lincoln Rd., Miami Beach, Florida/USA"


In [ ]:
from mllms import SimilarExamples
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])
bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)

few_shot_strategy = SimilarExamples(
    example_addresses=bzkopen_train["FullAddress"],
    example_labels=bzkopen_train,
    num_examples=3,
    labels_to_include=ENTITIES_TO_PREDICT,
    device=device,
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6398.27it/s]
MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:

def visualize_few_shots(address, example_strategy=None, prompt=None):
    if example_strategy is None:
        example_strategy = few_shot_strategy

    examples, metadatas = example_strategy.find_examples(address)
    has_breakdown = metadatas and "pattern_score" in metadatas[0]
    sec_key = next((k for k in ("token_score", "embedding_score") if metadatas and k in metadatas[0]), None)
    sec_label = {"token_score": "token", "embedding_score": "emb"}.get(sec_key, "")

    print(f"Query:   {address!r}")
    if metadatas and "query_pattern" in metadatas[0]:
        print(f"Pattern: {metadatas[0]['query_pattern']!r}")
    print()

    for m in metadatas:
        score_str = f"score={m['score']:.4f}"
        if has_breakdown:
            score_str += f"  (pattern={m['pattern_score']:.4f}, {sec_label}={m[sec_key]:.4f})"
        print(f"{score_str}")
        print(f"       address: {m['address']!r}")
        if "example_pattern" in m:
            print(f"       pattern: {m['example_pattern']!r}")
        labels = {k: v for k, v in m["example"].items() if v}
        print(f"       labels:  {labels}")
        print()

    if prompt is not None:
        print(prompt.make_prompt(address, examples))


In [10]:

# pattern hybrid strategy
from mllms import address_to_pattern

for addr in ["Karl-Marx-Straße 21/3, Berlin", "Krs. Breslau, Hauptstr. 4", "Haspe-Hagen"]:
    print(f"{addr!r:45s} → {address_to_pattern(addr)!r}")


'Karl-Marx-Straße 21/3, Berlin'               → 'WORD-WORD-STREET NUM/NUM, WORD'
'Krs. Breslau, Hauptstr. 4'                   → 'DIST WORD, WORD NUM'
'Haspe-Hagen'                                 → 'WORD-WORD'


In [11]:

# Instantiate the hybrid strategy (0.6 pattern + 0.4 token/Jaccard)
from mllms import PatternTokenSimilarExamples

pattern_strategy = PatternTokenSimilarExamples(
    example_addresses=bzkopen_train["FullAddress"],
    example_labels=bzkopen_train,
    num_examples=3,
    labels_to_include=ENTITIES_TO_PREDICT,
    pattern_weight=0.4,
)

# Visualise — the chart will show combined score + a breakdown panel (pattern vs token)
visualize_few_shots("Eichstetten Kr. Emmendingen", example_strategy=pattern_strategy)


Query:   'Eichstetten Kr. Emmendingen'
Pattern: 'WORD DIST WORD'

score=0.5200  (pattern=1.0000, token=0.2000)
       address: 'Petricken. Kr. Labian'
       pattern: 'WORD DIST WORD'
       labels:  {'City': 'Petricken', 'District': 'Labian'}

score=0.5200  (pattern=1.0000, token=0.2000)
       address: 'Hoffenheim Kr. Sinsheim'
       pattern: 'WORD DIST WORD'
       labels:  {'City': 'Hoffenheim', 'District': 'Sinsheim'}

score=0.5200  (pattern=1.0000, token=0.2000)
       address: 'Dieblich Kr. Koblenz'
       pattern: 'WORD DIST WORD'
       labels:  {'City': 'Dieblich', 'District': 'Koblenz'}

